In [1]:
from pathlib import Path
import sqlite3
import pandas as pd

PROJECT_ROOT = Path.cwd().parent

DB_PATH = PROJECT_ROOT / "database" / "sonoran_cycles.db"

print(DB_PATH)
print(DB_PATH.exists())

/Users/samuellettes/Desktop/Portfolio Project/Supply Chain Analyst/sonoran-cycles-analytics/database/sonoran_cycles.db
True


In [2]:
def run_query(query):
    """
    Runs a SQL query against the Sonoran Cycles SQLite database.
    """

    with sqlite3.connect(DB_PATH) as conn:
        return pd.read_sql_query(query, conn)

In [3]:
query = """
SELECT
    name AS table_name
FROM sqlite_master
WHERE type = 'table'
ORDER BY
    name;
"""

run_query(query)

,table_name
0,calendar
1,customers
2,daily_kpi_summary
3,daily_order_summary
4,forecast_accuracy_by_model
5,forecast_history
6,inventory_history
7,inventory_kpi_summary
8,model_performance_summary
9,monthly_sales_summary


In [11]:
query = """
SELECT
    model_name,
    category,
    SUM(requested_qty) AS requested_units,
    SUM(fulfilled_qty) AS fulfilled_units,
    SUM(backordered_qty) AS backordered_units,
    ROUND(SUM(extended_price), 2) AS booked_revenue,
    ROUND(SUM(fulfilled_revenue), 2) AS fulfilled_revenue,
    ROUND(SUM(gross_profit), 2) AS booked_gross_profit,
    ROUND(SUM(fulfilled_gross_profit), 2) AS fulfilled_gross_profit,
    ROUND(
        CAST(SUM(fulfilled_qty) AS FLOAT) / NULLIF(SUM(requested_qty), 0),
        3
    ) AS service_level,
    ROUND(
        CAST(SUM(backordered_qty) AS FLOAT) / NULLIF(SUM(requested_qty), 0),
        3
    ) AS backorder_rate
FROM sales_order_lines
GROUP BY
    model_name,
    category
ORDER BY
    booked_revenue DESC;

"""

model_performance = run_query(query)
model_performance

,model_name,category,requested_units,fulfilled_units,backordered_units,booked_revenue,fulfilled_revenue,booked_gross_profit,fulfilled_gross_profit,service_level,backorder_rate
0,Romero,Aggressive Trail,50939,26555,24384,1.693576e+08,88318116.48,15755153.58,8243638.38,0.521,0.479
1,Sabino,Trail,54251,27667,26584,1.637420e+08,83513838.87,24021772.74,12259140.39,0.510,0.490
2,Oracle,Enduro,35087,22371,12716,1.387805e+08,88465681.04,8277958.63,5258983.64,0.638,0.362
3,Catalina,Cross Country,29515,20296,9219,6.466980e+07,44426214.43,13063710.10,8939267.31,0.688,0.312
4,Sky Island,eMTB,12034,11166,868,5.891297e+07,54675733.93,1157470.65,1086080.17,0.928,0.072
5,Rincon,Downcountry,16641,14162,2479,4.226871e+07,35974524.12,5667662.73,4825913.22,0.851,0.149
6,Sonoita,Gravel,7533,7464,69,1.833556e+07,18172581.35,4629421.93,4591982.63,0.991,0.009


### Interpretation

This query summarizes demand, revenue, gross profit, backorders, and service level by bike model.

The main purpose is to identify which models are commercially important and whether high-revenue products are being fulfilled effectively. Models with strong booked revenue but weaker service levels represent potential planning risk because demand is present, but inventory availability is limiting shipment performance.

This table is useful for prioritizing demand planning attention by model rather than treating the entire catalog equally.

In [12]:
query = """
SELECT
    so.sales_channel,
    COUNT(DISTINCT so.sales_order_id) AS sales_orders,
    COUNT(DISTINCT so.customer_id) AS customers,
    SUM(sol.requested_qty) AS requested_units,
    SUM(sol.fulfilled_qty) AS fulfilled_units,
    SUM(sol.backordered_qty) AS backordered_units,
    ROUND(SUM(sol.extended_price), 2) AS booked_revenue,
    ROUND(SUM(sol.fulfilled_revenue), 2) AS fulfilled_revenue,
    ROUND(AVG(sol.unit_price), 2) AS average_unit_price,
    ROUND(
        CAST(SUM(sol.fulfilled_qty) AS FLOAT) / NULLIF(SUM(sol.requested_qty), 0),
        3
    ) AS service_level
FROM sales_orders so
JOIN sales_order_lines sol
    ON so.sales_order_id = sol.sales_order_id
GROUP BY
    so.sales_channel
ORDER BY
    booked_revenue DESC;

"""

channel_performance = run_query(query)
channel_performance

,sales_channel,sales_orders,customers,requested_units,fulfilled_units,backordered_units,booked_revenue,fulfilled_revenue,average_unit_price,service_level
0,Dealer,21477,35,188391,118326,70065,5.722776e+08,3.593848e+08,3038.85,0.628
1,DTC,15569,1,17609,11355,6254,8.378959e+07,5.416184e+07,4762.38,0.645


### Interpretation

This query compares performance across the Dealer and DTC sales channels.

Dealer orders typically represent larger wholesale demand with discounted pricing, while DTC orders represent smaller orders at higher unit prices. Comparing requested units, revenue, average unit price, and service level helps show how channel mix affects both demand volume and profitability.

This view is useful for understanding whether fulfillment issues are concentrated in one channel or spread across the business.

In [13]:
query = """
SELECT
    substr(so.order_date, 1, 7) AS order_month,
    so.sales_channel,
    COUNT(DISTINCT so.sales_order_id) AS sales_orders,
    SUM(sol.requested_qty) AS requested_units,
    SUM(sol.fulfilled_qty) AS fulfilled_units,
    SUM(sol.backordered_qty) AS backordered_units,
    ROUND(SUM(sol.extended_price), 2) AS booked_revenue,
    ROUND(SUM(sol.fulfilled_revenue), 2) AS fulfilled_revenue,
    ROUND(
        CAST(SUM(sol.fulfilled_qty) AS FLOAT) / NULLIF(SUM(sol.requested_qty), 0),
        3
    ) AS service_level
FROM sales_orders so
JOIN sales_order_lines sol
    ON so.sales_order_id = sol.sales_order_id
GROUP BY
    substr(so.order_date, 1, 7),
    so.sales_channel
ORDER BY
    order_month,
    so.sales_channel;
"""

monthly_trend = run_query(query)
monthly_trend.head()

,order_month,sales_channel,sales_orders,requested_units,fulfilled_units,backordered_units,booked_revenue,fulfilled_revenue,service_level
0,2022-01,DTC,199,218,218,0,1053582.00,1053582.00,1.000
1,2022-01,Dealer,260,2222,2173,49,6721621.14,6574225.05,0.978
2,2022-02,DTC,210,234,157,77,1149266.00,789743.00,0.671
3,2022-02,Dealer,287,2460,1620,840,7388016.47,4819158.72,0.659
4,2022-03,DTC,321,364,229,135,1733536.00,1116071.00,0.629


### Interpretation

This query shows monthly demand, revenue, and service level by sales channel.

The purpose is to confirm that the simulation produces seasonal demand patterns and to evaluate whether inventory availability keeps pace with demand during higher-volume months. Declining service levels during peak periods would suggest that reorder points, target stock levels, or supplier lead times may need adjustment.

This is one of the most important views for demand planning because it connects seasonality, sales volume, and fulfillment performance.

In [14]:
query = """
SELECT
    c.region,
    c.state,
    COUNT(DISTINCT so.sales_order_id) AS sales_orders,
    COUNT(DISTINCT so.customer_id) AS customers,
    SUM(sol.requested_qty) AS requested_units,
    ROUND(SUM(sol.extended_price), 2) AS booked_revenue,
    ROUND(SUM(sol.fulfilled_revenue), 2) AS fulfilled_revenue,
    ROUND(
        CAST(SUM(sol.fulfilled_qty) AS FLOAT) / NULLIF(SUM(sol.requested_qty), 0),
        3
    ) AS service_level
FROM sales_orders so
JOIN customers c
    ON so.customer_id = c.customer_id
JOIN sales_order_lines sol
    ON so.sales_order_id = sol.sales_order_id
WHERE so.sales_channel = 'Dealer'
GROUP BY
    c.region,
    c.state
ORDER BY
    booked_revenue DESC;
"""
regional_dealer_demand = run_query(query)
regional_dealer_demand

,region,state,sales_orders,customers,requested_units,booked_revenue,fulfilled_revenue,service_level
0,Southwest,Arizona,4254,6,37132,1.167572e+08,72529452.19,0.611
1,West Coast,California,3446,5,30280,9.709013e+07,65641637.10,0.669
2,Mountain,Colorado,2861,4,25179,7.379677e+07,45790590.64,0.625
3,West Coast,Washington,2125,3,18465,4.938144e+07,31807453.09,0.654
4,Southeast,North Carolina,1768,2,15529,4.823253e+07,27707224.36,0.572
5,Midwest,Minnesota,722,1,6331,1.944380e+07,11362367.38,0.585
6,Mountain,Utah,708,1,6140,1.881691e+07,10675486.74,0.566
7,Southeast,Tennessee,642,1,5533,1.702382e+07,9485389.92,0.557
8,Northeast,Vermont,768,1,6839,1.648348e+07,11757782.92,0.721
9,Midwest,Wisconsin,678,1,6001,1.462915e+07,10650790.28,0.734


### Interpretation

This query summarizes dealer demand and revenue by region and state.

The goal is to identify which geographic markets contribute the most revenue and unit demand. Strong regional concentration may indicate where dealer relationships, product availability, or regional replenishment planning deserve more attention.

This view is useful for territory planning, dealer account management, and evaluating whether inventory strategy supports the highest-demand markets.

In [7]:
query = """
SELECT
    sol.product_id,
    sol.sku,
    sol.model_name,
    sol.category,
    sol.size,
    sol.color,
    SUM(sol.requested_qty) AS requested_units,
    SUM(sol.fulfilled_qty) AS fulfilled_units,
    SUM(sol.backordered_qty) AS backordered_units,
    ROUND(SUM(sol.extended_price), 2) AS booked_revenue,
    ROUND(
        CAST(SUM(sol.backordered_qty) AS FLOAT) / NULLIF(SUM(sol.requested_qty), 0),
        3
    ) AS backorder_rate
FROM sales_order_lines sol
GROUP BY
    sol.product_id,
    sol.sku,
    sol.model_name,
    sol.category,
    sol.size,
    sol.color
HAVING
    SUM(sol.requested_qty) > 0
ORDER BY
    backordered_units DESC,
    backorder_rate DESC
LIMIT 25;
"""

worst_backorder_skus = run_query(query)
worst_backorder_skus

,product_id,sku,model_name,category,size,color,requested_units,fulfilled_units,backordered_units,booked_revenue,backorder_rate
0,1073,SON-SAB-C-M-IR,Sabino,Trail,M,Ironwood,3148,1213,1935,9504247.41,0.615
1,1074,SON-SAB-C-M-SG,Sabino,Trail,M,Saguaro,3125,1210,1915,9505719.09,0.613
2,1077,SON-SAB-C-M-CY,Sabino,Trail,M,Coyote,3140,1235,1905,9437102.01,0.607
3,1078,SON-SAB-C-M-SK,Sabino,Trail,M,Sky,3093,1190,1903,9334866.24,0.615
4,1076,SON-SAB-C-M-GR,Sabino,Trail,M,Granite,3114,1236,1878,9406472.67,0.603
5,1107,SON-ROM-C-M-CY,Romero,Aggressive Trail,M,Coyote,3035,1165,1870,10095938.10,0.616
6,1105,SON-ROM-C-M-SS,Romero,Aggressive Trail,M,Sunset,2985,1164,1821,9972513.84,0.610
7,1075,SON-SAB-C-M-SS,Sabino,Trail,M,Sunset,2970,1156,1814,8977018.05,0.611
8,1106,SON-ROM-C-M-GR,Romero,Aggressive Trail,M,Granite,2978,1218,1760,9886418.40,0.591
9,1108,SON-ROM-C-M-SK,Romero,Aggressive Trail,M,Sky,2958,1215,1743,9832452.78,0.589


### Interpretation

This query identifies the SKUs with the highest backordered units.

These products represent demand that was booked but could not be fully fulfilled from available inventory. High backorders may indicate under-forecasting, insufficient target stock, poor SKU-level inventory balance, or supplier lead-time issues.

This table helps move the analysis from model-level performance to actionable SKU-level planning decisions.

In [8]:
query = """
SELECT
    product_id,
    sku,
    model_name,
    category,
    size,
    color,
    ROUND(average_available, 2) AS average_available,
    minimum_available,
    ending_available,
    stockout_days,
    ROUND(stockout_rate, 3) AS stockout_rate
FROM inventory_kpi_summary
ORDER BY
    stockout_days DESC,
    stockout_rate DESC,
    average_available ASC
LIMIT 25;
"""

worst_stockout_skus = run_query(query)
worst_stockout_skus

,product_id,sku,model_name,category,size,color,average_available,minimum_available,ending_available,stockout_days,stockout_rate
0,1074,SON-SAB-C-M-SG,Sabino,Trail,M,Saguaro,11.63,0,0,896,0.613
1,1076,SON-SAB-C-M-GR,Sabino,Trail,M,Granite,12.18,0,0,887,0.607
2,1106,SON-ROM-C-M-GR,Romero,Aggressive Trail,M,Granite,12.73,0,23,861,0.589
3,1073,SON-SAB-C-M-IR,Sabino,Trail,M,Ironwood,12.47,0,0,860,0.589
4,1077,SON-SAB-C-M-CY,Sabino,Trail,M,Coyote,13.05,0,1,860,0.589
5,1103,SON-ROM-C-M-IR,Romero,Aggressive Trail,M,Ironwood,13.38,0,0,856,0.586
6,1108,SON-ROM-C-M-SK,Romero,Aggressive Trail,M,Sky,12.84,0,10,851,0.582
7,1104,SON-ROM-C-M-SG,Romero,Aggressive Trail,M,Saguaro,12.83,0,8,842,0.576
8,1078,SON-SAB-C-M-SK,Sabino,Trail,M,Sky,13.14,0,16,837,0.573
9,1107,SON-ROM-C-M-CY,Romero,Aggressive Trail,M,Coyote,13.28,0,0,834,0.571


### Interpretation

This query identifies SKUs with the most days at or below zero available inventory.

Stockout days are a direct measure of inventory availability risk. A SKU may not have the highest total backorders, but frequent stockouts still indicate weak inventory positioning and possible lost sales exposure.

This view is useful for identifying products that may need higher reorder points, higher target stock, or closer replenishment monitoring.

In [9]:
query = """
SELECT
    supplier_id,
    supplier_name,
    purchase_orders,
    open_purchase_orders,
    received_purchase_orders,
    ordered_units,
    received_units,
    open_units,
    ROUND(average_lead_time_days, 1) AS average_lead_time_days,
    ROUND(receipt_rate, 3) AS receipt_rate
FROM supplier_performance_summary
ORDER BY
    open_units DESC,
    ordered_units DESC;
"""

supplier_exposure = run_query(query)
supplier_exposure

,supplier_id,supplier_name,purchase_orders,open_purchase_orders,received_purchase_orders,ordered_units,received_units,open_units,average_lead_time_days,receipt_rate
0,S001,Merida Industry,2029,68,1961,124704,120531,4173,55.0,0.967
1,S002,Ideal Bike Corp,126,2,124,7684,7561,123,45.0,0.984


### Interpretation

This query summarizes purchase order activity by supplier.

The purpose is to understand which suppliers are responsible for the largest replenishment volume and how much open purchase order exposure remains. Open units indicate inbound supply that has been ordered but not yet received.

In the current simulation, purchase orders are tied to finished-goods replenishment. Component suppliers such as drivetrain, suspension, tire, and wheel suppliers are retained in the supplier master for possible future BOM-level expansion.

In [10]:
query = """
SELECT
    model_name,
    category,
    actual_qty,
    forecast_qty,
    absolute_error,
    forecast_error,
    ROUND(wape, 3) AS wape,
    ROUND(bias_pct, 3) AS bias_pct,
    ROUND(forecast_accuracy, 3) AS forecast_accuracy
FROM forecast_accuracy_by_model
ORDER BY
    wape DESC;
"""

forecast_accuracy = run_query(query)
forecast_accuracy

,model_name,category,actual_qty,forecast_qty,absolute_error,forecast_error,wape,bias_pct,forecast_accuracy
0,Sonoita,Gravel,7265,7148,4057,117,0.558,0.016,0.442
1,Sky Island,eMTB,11464,11434,5440,30,0.475,0.003,0.525
2,Rincon,Downcountry,15880,15854,6700,26,0.422,0.002,0.578
3,Catalina,Cross Country,28225,27918,8741,307,0.310,0.011,0.690
4,Oracle,Enduro,33376,33426,9530,-50,0.286,-0.001,0.714
5,Romero,Aggressive Trail,48393,48623,11716,-230,0.242,-0.005,0.758
6,Sabino,Trail,51665,51664,11961,1,0.232,0.000,0.768


### Interpretation

This query compares forecasted demand to actual requested demand by model.

WAPE shows the size of the forecast error relative to actual demand, while bias shows whether the forecast tends to overstate or understate demand. A positive bias means actual demand was higher than forecast, while a negative bias means forecasted demand was higher than actual demand.

This view is useful for identifying which models are harder to forecast and where the demand planning process may need refinement.

In [15]:
query = """
SELECT
    forecast_month,
    SUM(actual_qty) AS actual_qty,
    SUM(forecast_qty) AS forecast_qty,
    SUM(absolute_error) AS absolute_error,
    SUM(forecast_error) AS forecast_error,
    ROUND(
        CAST(SUM(absolute_error) AS FLOAT) / NULLIF(SUM(actual_qty), 0),
        3
    ) AS wape,
    ROUND(
        CAST(SUM(forecast_error) AS FLOAT) / NULLIF(SUM(actual_qty), 0),
        3
    ) AS bias_pct
FROM forecast_history
GROUP BY
    forecast_month
ORDER BY
    forecast_month;
"""
forecast_bias = run_query(query)
forecast_bias

,forecast_month,actual_qty,forecast_qty,absolute_error,forecast_error,wape,bias_pct
0,2022-04-01,5114,5564,1464,-450,0.286,-0.088
1,2022-05-01,6292,6085,1561,207,0.248,0.033
2,2022-06-01,6109,6038,1411,71,0.231,0.012
3,2022-07-01,5129,5222,1479,-93,0.288,-0.018
4,2022-08-01,4915,4608,1313,307,0.267,0.062
5,2022-09-01,4111,4211,1244,-100,0.303,-0.024
6,2022-10-01,3337,3474,1197,-137,0.359,-0.041
7,2022-11-01,3917,3730,1219,187,0.311,0.048
8,2022-12-01,2865,2784,995,81,0.347,0.028
9,2023-01-01,2515,2601,914,-86,0.363,-0.034


### Interpretation

This query evaluates forecast accuracy and bias by forecast month.

The goal is to identify whether forecast error is concentrated in specific months or seasons. If certain months show consistent under-forecasting or over-forecasting, the seasonality assumptions may need to be adjusted.

This is useful for evaluating whether the forecasting method captures seasonal demand shifts well enough for planning purposes.

In [16]:
query = """
WITH product_performance AS (
    SELECT
        sol.product_id,
        sol.sku,
        sol.model_name,
        sol.category,
        sol.size,
        sol.color,
        SUM(sol.requested_qty) AS requested_units,
        SUM(sol.fulfilled_qty) AS fulfilled_units,
        SUM(sol.backordered_qty) AS backordered_units,
        ROUND(SUM(sol.extended_price), 2) AS booked_revenue,
        ROUND(
            CAST(SUM(sol.fulfilled_qty) AS FLOAT) / NULLIF(SUM(sol.requested_qty), 0),
            3
        ) AS service_level
    FROM sales_order_lines sol
    GROUP BY
        sol.product_id,
        sol.sku,
        sol.model_name,
        sol.category,
        sol.size,
        sol.color
)

SELECT
    product_id,
    sku,
    model_name,
    category,
    size,
    color,
    requested_units,
    fulfilled_units,
    backordered_units,
    booked_revenue,
    service_level
FROM product_performance
WHERE
    requested_units >= 50
    AND service_level < 0.90
ORDER BY
    booked_revenue DESC,
    service_level ASC;
"""
high_demand_products_weak_service = run_query(query)
high_demand_products_weak_service

,product_id,sku,model_name,category,size,color,requested_units,fulfilled_units,backordered_units,booked_revenue,service_level
0,1107,SON-ROM-C-M-CY,Romero,Aggressive Trail,M,Coyote,3035,1165,1870,10095938.10,0.384
1,1105,SON-ROM-C-M-SS,Romero,Aggressive Trail,M,Sunset,2985,1164,1821,9972513.84,0.390
2,1106,SON-ROM-C-M-GR,Romero,Aggressive Trail,M,Granite,2978,1218,1760,9886418.40,0.409
3,1108,SON-ROM-C-M-SK,Romero,Aggressive Trail,M,Sky,2958,1215,1743,9832452.78,0.411
4,1104,SON-ROM-C-M-SG,Romero,Aggressive Trail,M,Saguaro,2941,1212,1729,9786077.70,0.412
...,...,...,...,...,...,...,...,...,...,...,...
111,1025,SON-CAT-C-XL-IR,Catalina,Cross Country,XL,Ironwood,729,597,132,1597573.74,0.819
112,1009,SON-CAT-C-S-SS,Catalina,Cross Country,S,Sunset,720,596,124,1574645.69,0.828
113,1028,SON-CAT-C-XL-GR,Catalina,Cross Country,XL,Granite,724,649,75,1567552.84,0.896
114,1010,SON-CAT-C-S-GR,Catalina,Cross Country,S,Granite,693,599,94,1517507.01,0.864


### Interpretation

This query highlights products with meaningful demand but service levels below the desired threshold.

These SKUs are especially important because they combine commercial importance with fulfillment weakness. They are stronger candidates for planning intervention than low-demand SKUs with poor service levels.

This view helps prioritize where inventory policy changes could have the greatest business impact.

In [17]:
query = """
SELECT
    date AS summary_date,
    sales_orders,
    requested_units,
    fulfilled_units,
    backordered_units,
    ROUND(service_level, 3) AS service_level,
    ROUND(backorder_rate, 3) AS backorder_rate,
    purchase_orders_created,
    purchase_units_ordered,
    purchase_orders_received,
    purchase_units_received,
    total_on_hand_units,
    out_of_stock_skus
FROM daily_kpi_summary
ORDER BY
    summary_date;
"""
daily_service_levels = run_query(query)
daily_service_levels

,summary_date,sales_orders,requested_units,fulfilled_units,backordered_units,service_level,backorder_rate,purchase_orders_created,purchase_units_ordered,purchase_orders_received,purchase_units_received,total_on_hand_units,out_of_stock_skus
0,2022-01-01,11,41,41,0,1.000,0.000,0,0,0,0,8359,0
1,2022-01-02,9,13,13,0,1.000,0.000,0,0,0,0,8346,0
2,2022-01-03,18,130,130,0,1.000,0.000,0,0,0,0,8216,0
3,2022-01-04,19,143,143,0,1.000,0.000,0,0,0,0,8073,0
4,2022-01-05,17,114,114,0,1.000,0.000,1,64,0,0,7959,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1456,2025-12-27,13,51,34,17,0.667,0.333,0,0,1,60,6763,25
1457,2025-12-28,9,21,13,8,0.619,0.381,0,0,1,60,6810,25
1458,2025-12-29,16,116,95,21,0.819,0.181,1,60,2,122,6837,27
1459,2025-12-30,19,112,79,33,0.705,0.295,0,0,1,61,6819,27


### Interpretation

This query shows daily sales, fulfillment, purchasing, and inventory KPIs over time.

The goal is to observe how service level, backorders, purchase order creation, receipts, and total inventory move together. This helps explain whether fulfillment problems are temporary timing issues or persistent inventory planning problems.

This view is useful for diagnosing the relationship between demand spikes, replenishment timing, and inventory availability.

In [18]:
query = """
SELECT
    c.dealer_tier,
    COUNT(DISTINCT so.customer_id) AS customers,
    COUNT(DISTINCT so.sales_order_id) AS sales_orders,
    SUM(sol.requested_qty) AS requested_units,
    ROUND(SUM(sol.extended_price), 2) AS booked_revenue,
    ROUND(SUM(sol.fulfilled_revenue), 2) AS fulfilled_revenue,
    ROUND(
        CAST(SUM(sol.fulfilled_qty) AS FLOAT) / NULLIF(SUM(sol.requested_qty), 0),
        3
    ) AS service_level
FROM sales_orders so
JOIN customers c
    ON so.customer_id = c.customer_id
JOIN sales_order_lines sol
    ON so.sales_order_id = sol.sales_order_id
WHERE
    so.sales_channel = 'Dealer'
GROUP BY
    c.dealer_tier
ORDER BY
    booked_revenue DESC;
"""
dealer_tier_performance = run_query(query)
dealer_tier_performance

,dealer_tier,customers,sales_orders,requested_units,booked_revenue,fulfilled_revenue,service_level
0,Gold,16,11282,98934,2.984710e+08,1.837793e+08,0.617
1,Platinum,5,5323,46567,1.395947e+08,8.981254e+07,0.635
2,Silver,14,4872,42890,1.342119e+08,8.579298e+07,0.646


### Interpretation

This query compares demand, revenue, and service level by dealer tier.

The purpose is to evaluate whether higher-tier dealers are driving a meaningful share of revenue and whether they are being served effectively. If key dealer tiers have weaker service levels, that may indicate a risk to important customer relationships.

This view is useful for account prioritization and for understanding whether inventory availability supports the most strategically important dealers.

## Summary

The SQL analysis shows how the simulated supply chain can be investigated through a relational database rather than only through Python DataFrames.

The main analytical themes are:

- Revenue and demand are concentrated by model, channel, dealer region, and SKU.
- Fulfillment performance can be evaluated through service level, backorder rate, and stockout days.
- Supplier exposure can be monitored through open purchase orders and inbound units.
- Forecast quality can be measured using WAPE and bias.
- SKU-level exceptions help identify where inventory policy may need adjustment.

Together, these queries demonstrate how SQL can support demand planning, inventory analysis, supplier monitoring, and executive dashboard development.